# EX04 — Token Efficiency & Cost Analysis

**Project:** Reverse Engineering with Grphify + CrewAI  
**Target:** cookiecutter (18 Python files, 269 nodes, 504 edges)  
**Key idea:** agents read the *graph* first, not the full source — this saves tokens.

This notebook analyses token usage and cost across the pipeline stages.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

stats = json.loads(Path("../results/token_stats.json").read_text())
model = stats["model"]
pricing = stats["pricing_usd_per_million"]
stages = stats["stages"]

print(f"Model: {model}")
print(f"Pricing: ${pricing['input']}/M input  ${pricing['output']}/M output")
print()
for name, data in stages.items():
    total = data["input_tokens"] + data["output_tokens"]
    print(f"{name:<30}  {total:>7,} tokens  — {data['description']}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

stage_names = [s.replace("_", "\n") for s in stages]
totals = [v["input_tokens"] + v["output_tokens"] for v in stages.values()]
colors = ["#e74c3c" if "naive" in s else "#2ecc71" for s in stages]

axes[0].bar(stage_names, totals, color=colors)
axes[0].set_title("Token Usage per Stage", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Tokens")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
axes[0].tick_params(axis="x", labelsize=9)

compare_labels = ["Naive\n(all source)", "Graph-guided\n(agents)"]
compare_vals = [stats["naive_tokens_estimated"], stats["graph_guided_tokens_estimated"]]
bars = axes[1].bar(compare_labels, compare_vals, color=["#e74c3c", "#2ecc71"], width=0.4)
axes[1].set_title("Naive vs Graph-Guided Token Usage", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Tokens")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for bar, val in zip(bars, compare_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f"{val:,}", ha="center", va="bottom", fontweight="bold")
axes[1].annotate(f"{stats['savings_percent']}% saving", xy=(0.5, 0.85),
                 xycoords="axes fraction", ha="center", fontsize=12,
                 color="#27ae60", fontweight="bold")

plt.suptitle("EX04 — Token Efficiency Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("token_efficiency.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
p_in  = pricing["input"]  / 1_000_000
p_out = pricing["output"] / 1_000_000

print(f"{'Stage':<28} {'Input':>8} {'Output':>8} {'Cost USD':>12}")
print("-" * 62)
total_cost = 0.0
for name, data in stages.items():
    in_tok  = data["input_tokens"]
    out_tok = data["output_tokens"]
    cost    = in_tok * p_in + out_tok * p_out
    total_cost += cost
    print(f"{name:<28} {in_tok:>8,} {out_tok:>8,} ${cost:>11.6f}")
print("-" * 62)
print(f"{'TOTAL':<28} {stats['total_input_tokens']:>8,} {stats['total_output_tokens']:>8,} ${total_cost:>11.6f}")
print()
print(f"Naive baseline cost:       ${stats['naive_cost_usd']:.4f}")
print(f"Graph-guided total cost:   ${stats['total_cost_usd']:.4f}")
print(f"Token savings:             {stats['savings_percent']}% fewer tokens in agent prompts")


## Interpretation

| Metric | Value |
|--------|-------|
| Naive token cost (send all source to LLM) | 23,537 tokens |
| Graph-guided agent prompt tokens | 645 tokens |
| **Token savings** | **97.3 %** |
| Total pipeline cost | ~$0.003 |

**Why the graph saves tokens:** Instead of feeding all 18 Python files into the LLM context,
the Graph Navigator agent reads only `graph.json` + `hot.md` (~645 tokens) to identify
the top hubs. The architect then drills into only the specific file that matters.

**Cost efficiency:** At Gemini 2.5 Flash pricing ($0.075/M input, $0.30/M output),
the entire pipeline run costs approximately **$0.003** — less than a fraction of a cent.

**Takeaway:** Graph-guided context injection replaces brute-force RAG (send everything)
with targeted, graph-derived context injection — a 97.3% reduction in agent prompt tokens.
